# Notebook 05: Integración del Módulo LLM y Generación de Recomendaciones

## Objetivo
Simular el flujo completo del backend: desde la carga de datos hasta la generación de recomendaciones
estratégicas mediante el LLM local (Ollama). Este notebook demuestra:
1. La construcción del **Contrato de Datos** (payload JSON) que alimenta tanto la UI como el LLM.
2. La **Ingeniería de Prompts Dinámica** que adapta las recomendaciones al rol (PM/PMO) y alcance (Sprint/Proyecto).
3. La inferencia local con métricas de latencia (RNF-02) y tolerancia a fallos (RNF-07).
4. La validación de calidad mediante **LLM-as-a-judge**.

**Nota sobre privacidad:** Todo el procesamiento se ejecuta localmente mediante Ollama (Privacy by Design, RNF-05).

In [1]:
# ==============================================================================
# 1. IMPORTACIÓN DE LIBRERÍAS Y CONFIGURACIÓN INICIAL
# ==============================================================================
import json
import time
import ollama # Ejecución local (Stateless, Privacy by Design RNF-05)

LLM_MODEL = 'llama3' 

print(f"✅ Entorno de IA Generativa inicializado. Modelo: {LLM_MODEL}")

✅ Entorno de IA Generativa inicializado. Modelo: llama3


## 1. Configuración y carga de datos
Se define el alcance del análisis (`Sprint` o `Proyecto`) y el rol del usuario (`Project Manager` o `PMO`).
Estos parámetros simulan la entrada que recibiría la API desde la interfaz.
El filtrado en memoria selecciona únicamente los datos del alcance elegido.

In [ ]:
# ==============================================================================
# 2. DEFINICIÓN DEL CONTEXTO (Motor Híbrido Bidimensional y UI Dinámica)
# ==============================================================================
import json
import pandas as pd
import joblib

# ------------------------------------------------------------------------------
# A. CONFIGURACIÓN INICIAL Y CARGA DE DATOS (Simulador de Backend)
# ------------------------------------------------------------------------------
# El parámetro 'alcance_analisis' bifurcará la forma en la que se agrupan los 
# datos más adelante, simulando la ingesta de la API.
alcance_analisis = "Sprint" # Opciones: "Sprint" (Operativo) o "Proyecto" (Estratégico)
rol_usuario = "Project Manager" # Opciones: "Project Manager" o "PMO"

ruta_datos = '../data/synthetic/proyecto_completo.csv'
df_proyecto = pd.read_csv(ruta_datos)

# Filtrado inicial en memoria según el alcance
if alcance_analisis == "Sprint":
    df_analisis = df_proyecto[df_proyecto['Sprint_ID'] == 3].copy()
else:
    df_analisis = df_proyecto.copy()

# ------------------------------------------------------------------------------


## 2. Preprocesamiento y prevención de Data Leakage
Se replica la misma estrategia de eliminación de variables del notebook 04 para garantizar que
los datos de entrada a los modelos sean coherentes con los usados durante el entrenamiento.

In [ ]:
# B. PREPROCESAMIENTO: PREVENCIÓN DE DATA LEAKAGE Y PREPARACIÓN DE FEATURES
# ------------------------------------------------------------------------------
# 1. Limpieza de variables globales (comunes a ambos modelos)
cols_to_drop_global = [
    'Issue_Key', 'Project_ID', 'Project_Name', 'Sprint_ID', 
    'Resolution_Time_Minutes', 'Target_Retraso', 'Target_Riesgo'
]
X_ret = df_analisis.drop(columns=cols_to_drop_global, errors='ignore')

# 2. Limpieza específica para el modelo de riesgos (evitar que el modelo haga trampa)
cols_leakage_riesgo = [
    'Blocker_Count', 'Story_Point_Changed_After_Estimation', 'Description_Changed_After_Estimation'
]
X_rsg = X_ret.drop(columns=cols_leakage_riesgo, errors='ignore')

# ------------------------------------------------------------------------------


## 3. Inferencia de IA: Ejecución de los modelos XGBoost
Se cargan los modelos serializados y se ejecuta la predicción de probabilidad de retraso y riesgo
para cada tarea del alcance seleccionado. Ambas dimensiones se calculan siempre a nivel individual.

In [ ]:
# C. INFERENCIA DE IA: EJECUCIÓN DE MODELOS XGBOOST
# ------------------------------------------------------------------------------
# Independientemente del alcance, SIEMPRE se calculan ambas dimensiones 
# (Retrasos y Riesgos) a nivel de tarea individual.
dict_retrasos = joblib.load('../backend/models/modelo_retrasos_xgb.pkl')
pipeline_retraso = dict_retrasos['pipeline']
pipeline_riesgo = joblib.load('../backend/models/modelo_riesgos_xgb.pkl')

df_analisis['Prob_Retraso'] = pipeline_retraso.predict_proba(X_ret)[:, 1]
df_analisis['Prob_Riesgo'] = pipeline_riesgo.predict_proba(X_rsg)[:, 1]

# ------------------------------------------------------------------------------


## 4. Identificación de tareas críticas
Se filtran las **Top 3 tareas** en cada dimensión:
- **Riesgo:** Tareas con probabilidad > 55% o bloqueos activos, ordenadas por gravedad (semaforo rojo/amarillo/verde).
- **Retraso:** Tareas con mayor probabilidad de incumplimiento del cronograma.

In [ ]:
# D. FILTRADO A NIVEL OPERATIVO: TOP TAREAS CRÍTICAS
# ------------------------------------------------------------------------------
# Identificamos las tareas individuales que requieren atención inmediata.

# D.1. Dimensión Riesgo (Bloqueos activos y fallos estructurales)
cond_riesgo = (df_analisis['Prob_Riesgo'] > 0.55) | (df_analisis['Blocker_Count'] > 0)
top_riesgos_df = df_analisis[cond_riesgo].sort_values(by=['Blocker_Count', 'Prob_Riesgo'], ascending=[False, False]).head(3)

lista_top_riesgos = []
for _, row in top_riesgos_df.iterrows():
    prob_riesgo = float(row['Prob_Riesgo'])
    bloqueos = int(row.get('Blocker_Count', 0))
    
    # Lógica de Gravedad para renderizar el semáforo en la Interfaz (UI)
    if prob_riesgo >= 0.75 or bloqueos > 1:
        gravedad_semaforo = "🔴 Alto"
    elif prob_riesgo >= 0.55 or bloqueos == 1:
        gravedad_semaforo = "🟡 Medio"
    else:
        gravedad_semaforo = "🟢 Bajo"

    lista_top_riesgos.append({
        "Issue_Key": str(row.get('Issue_Key', 'N/A')),
        "Issue_Type": str(row.get('Issue_Type', 'N/A')),
        "Title": str(row.get('Title', 'Sin título')),        # ← CAMBIA Summary por Title
        "Blocker_Count": bloqueos,
        "Prob_Riesgo": round(prob_riesgo, 2),
        "Gravedad": gravedad_semaforo
    })

# D.2. Dimensión Retraso (Desviaciones de tiempo/esfuerzo)
cond_retraso = df_analisis['Prob_Retraso'] > 0.55
top_retrasos_df = df_analisis[cond_retraso].sort_values(by='Prob_Retraso', ascending=False).head(3)

lista_top_retrasos = []
for _, row in top_retrasos_df.iterrows():
        lista_top_retrasos.append({
        "Issue_Key": str(row.get('Issue_Key', 'N/A')),
        "Issue_Type": str(row.get('Issue_Type', 'N/A')),
        "Title": str(row.get('Title', 'Sin título')),        # ← CAMBIA Summary por Title
        "Prob_Retraso": round(float(row['Prob_Retraso']), 2)
    })

# ------------------------------------------------------------------------------


## 5. Agregación estratégica y cálculo de KPIs
Se calculan los indicadores clave que alimentarán el dashboard y el contexto del LLM:
- KPIs transversales: riesgo medio, retraso medio, tareas bloqueadas, semáforo global.
- Métricas de impacto: esfuerzo total comprometido en tareas críticas.
- Tendencias temporales: evolución agrupada por Sprint (Proyecto) o por día (Sprint).

In [ ]:
# E. AGREGACIÓN ESTRATÉGICA Y CÁLCULO DE KPIs
# ------------------------------------------------------------------------------

# E.1. KPIs Transversales (Aplican igual tanto a Sprint como a Proyecto)
kpi_riesgo_medio = float(df_analisis['Prob_Riesgo'].mean())
kpi_retraso_medio = float(df_analisis['Prob_Retraso'].mean())
tareas_bloqueadas_total = int(df_analisis['Blocker_Count'].sum())
total_story_points = float(df_analisis.get('Story_Point', pd.Series([0])).sum())
semaforo_global = "Rojo" if kpi_riesgo_medio >= 0.60 else "Amarillo" if kpi_riesgo_medio >= 0.30 else "Verde"

if 'Status' in df_analisis.columns:
    tareas_cerradas = len(df_analisis[df_analisis['Status'].str.lower().isin(['closed', 'resolved'])])
elif 'Sprint_State' in df_analisis.columns and df_analisis['Sprint_State'].iloc[0] == 'CLOSED':
    tareas_cerradas = len(df_analisis[df_analisis['Resolution_Time_Minutes'] > 0])
else:
    tareas_cerradas = 0
tasa_completado = round((tareas_cerradas / max(1, len(df_analisis))) * 100, 2)

# E.2. Métricas de Impacto de Negocio (Para justificar decisiones ante la PMO/Cliente)
tareas_criticas = df_analisis[(df_analisis['Prob_Riesgo'] > 0.7) | (df_analisis['Prob_Retraso'] > 0.7) | (df_analisis['Blocker_Count'] > 0)]
esfuerzo_en_riesgo = float(tareas_criticas.get('Story_Point', tareas_criticas.get('Total_Effort_Minutes', pd.Series([0]))).sum())

# E.3. Tendencias Temporales (BIFURCACIÓN POR ALCANCE)
evolucion_riesgo = {}
evolucion_retraso = {}

if alcance_analisis == "Proyecto" and 'Sprint_ID' in df_proyecto.columns:
    # Alcance Estratégico: Agrupación macro por iteraciones completas (Sprints)
    evolucion_riesgo = df_proyecto.groupby('Sprint_ID')['Prob_Riesgo'].mean().round(2).to_dict()
    evolucion_retraso = df_proyecto.groupby('Sprint_ID')['Prob_Retraso'].mean().round(2).to_dict()
else: 
    # Alcance Operativo: Agrupación micro día a día dentro del propio Sprint
    if 'Created_Date' in df_analisis.columns:
        df_analisis['Dia'] = pd.to_datetime(df_analisis['Created_Date']).dt.date
        evolucion_riesgo = df_analisis.groupby(df_analisis['Dia'].astype(str))['Prob_Riesgo'].mean().round(2).to_dict()
        evolucion_retraso = df_analisis.groupby(df_analisis['Dia'].astype(str))['Prob_Retraso'].mean().round(2).to_dict()
    else:
        # Sin fechas disponibles: no se inyectan datos ficticios
        evolucion_riesgo = {}
        evolucion_retraso = {}

# ------------------------------------------------------------------------------


## 6. Ensamblaje del Contrato de Datos (Payload JSON)
Se consolida toda la información en un payload estructurado con tres bloques:
- `UI_Header_KPIs`: Métricas para las tarjetas superiores del dashboard.
- `UI_Tab_1_Estado`: Datos para gráficos y alertas (Tab de métricas).
- `LLM_Tab_2_Contexto`: Contexto analítico inyectado en el System Prompt del LLM.

In [2]:
# F. ENSAMBLAJE DEL PAYLOAD JSON (Contrato de Datos para UI y LLM)
# ------------------------------------------------------------------------------
payload_sistema = {
    "Configuracion": {
        "Alcance": alcance_analisis,
        "Rol": rol_usuario
    },
    "UI_Header_KPIs": {
        "Total_Tareas": int(len(df_analisis)),
        "Tasa_Completado_Pct": tasa_completado,
        "Esfuerzo_Total": total_story_points,
        "Tareas_Bloqueadas_Activas": tareas_bloqueadas_total,
        "Riesgo_Promedio": round(kpi_riesgo_medio, 2),
        "Retraso_Promedio": round(kpi_retraso_medio, 2)
    },
    "UI_Tab_1_Estado": {
        "Semaforo_Riesgo_Global": semaforo_global,
        "Alerta_Retraso_Global": "Activada" if kpi_retraso_medio > 0.50 else "Desactivada",
        "Grafico_Riesgo_por_Tipo": df_analisis.groupby('Issue_Type')['Prob_Riesgo'].mean().round(2).to_dict(),
        "Grafico_Evolucion_Riesgo": evolucion_riesgo 
    },
    "LLM_Tab_2_Contexto": {
        "Métricas_Globales_Negocio": {
            "Riesgo_General": round(kpi_riesgo_medio, 2),
            "Retraso_General": round(kpi_retraso_medio, 2),
            "Total_Bloqueos_Activos": tareas_bloqueadas_total,
            "Esfuerzo_Total_Comprometido_En_Riesgo": esfuerzo_en_riesgo
        },
        "Tendencias": {
            "Evolucion_Riesgo": evolucion_riesgo,
            "Evolucion_Retraso": evolucion_retraso
        },
        "Top_Tareas_Riesgo_Bloqueos": lista_top_riesgos,
        "Top_Tareas_Retraso_Cronograma": lista_top_retrasos
    }
}

json_completo_str = json.dumps(payload_sistema, indent=2, ensure_ascii=False)
print("Contrato de Datos Generado exitosamente:\n")
print(json_completo_str)

## 7. Ingeniería de Prompts y mitigación de alucinaciones (RT-03)
Se define `build_prompt`, la función que construye dinámicamente el System Prompt en función del
**rol** (PM táctico vs PMO estratégico) y el **alcance** (operativo vs global).
Incluye reglas estrictas anti-alucinación y de interpretación de probabilidades.

In [3]:
# ==============================================================================
# 3. INGENIERÍA DE PROMPTS Y MITIGACIÓN DE RIESGOS (RT-03)
# ==============================================================================
import json

def build_prompt(context_data_llm: dict, user_role: str, scope: str) -> tuple[str, str]:
    # Convertimos el diccionario de contexto específico para el LLM a string
    context_str = json.dumps(context_data_llm, indent=2, ensure_ascii=False)
    
    # 1. Instrucción de Alcance (TUS DEFINICIONES ORIGINALES)
    if scope == "Sprint":
        focus = "ALCANCE SPRINT: Evalúa las predicciones de riesgo y retraso correspondientes a las tareas de esta iteración única. Tu prioridad es mitigar el impacto operativo inmediato analizando la carga de trabajo y los bloqueos diarios activos."
    else:
        focus = "ALCANCE PROYECTO: Evalúa las predicciones de riesgo y retraso agregadas de múltiples iteraciones. Tu prioridad es identificar tendencias de desviación a largo plazo, evaluar la salud global del proyecto y detectar patrones de bloqueo recurrentes."

    # 2. Instrucción de Rol (TUS DEFINICIONES ORIGINALES)
    if user_role == "Project Manager":
        instruccion_rol = "ROL PM: Eres un consultor táctico. Genera recomendaciones operativas a corto plazo (reasignación de tareas, resolución de bloqueos, repriorización del backlog, pair programming). Usa un tono directo y orientado a la ejecución del equipo."
    else: # PMO
        instruccion_rol = "ROL PMO: Eres un consultor estratégico. Genera recomendaciones estructurales (cambios metodológicos, reajuste de presupuesto, auditoría de procesos corporativos, balanceo de capacidad técnica). Usa un tono analítico, formal y directivo."

    # 3. Ensamblaje del System Prompt Maestro (Con parches anti-alucinaciones)
    system_prompt = f"""
    Eres un Sistema Inteligente de Apoyo a la Decisión para la gestión ágil de proyectos.
    
    {instruccion_rol}
    {focus}
    
    REGLAS ESTRICTAS DE NEGOCIO:
    1. IDIOMA ESTRICTO: Responde ÚNICA Y EXCLUSIVAMENTE en Español. No uses términos en inglés salvo los propios del contexto (ej. Issue_Key).
    2. ABORDA AMBAS DIMENSIONES: Propón soluciones tanto para RETRASO como para RIESGO. Agrupa tareas similares en una misma recomendación.
    3. JUSTIFICACIÓN Y CERO ALUCINACIONES: Justifica cada recomendación citando los valores del contexto. NUNCA inventes Issue_Keys, títulos ni métricas.
    4. NOMBRA LAS TAREAS: Cita siempre Issue_Key y Title juntos (ej. "PRO-301 'Error de validación en formulario'").
    5. ESTRUCTURA: Devuelve exactamente 3 recomendaciones en viñetas. Cada viñeta debe citar al menos una tarea distinta. Sin introducciones ni saludos. Empieza con •.

    <reglas_interpretacion>
    - NUNCA escribas probabilidades como decimales. 1.0 → "100%", 0.63 → "63%", 0.80 → "80%". Sin excepciones. Aplica a Prob_Riesgo, Prob_Retraso, Riesgo_General y Retraso_General.
    - Un valor de Prob_Riesgo de 0.06 significa riesgo BAJO. Un valor de 0.94 significa riesgo ALTO.
    - Una probabilidad de retraso superior al 50% es ALTA, no baja. Superior al 75% es CRÍTICA.
    - Toda viñeta debe mencionar el porcentaje explícitamente, nunca solo la etiqueta "Alto" o "Medio".
    - La gravedad de cada tarea viene en el campo "Gravedad". Cópialo tal cual, no lo recalcules.
    - Si Blocker_Count > 0, la tarea es urgente independientemente de su probabilidad.
    </reglas_interpretacion>

    <datos_ml>
    {context_str}
    </datos_ml>
    """
    
    user_prompt = "Redacta el reporte de recomendaciones estratégicas en perfecto español basándote estrictamente en las predicciones proporcionadas."
    return system_prompt, user_prompt

# Probamos la generación del prompt pasándole solo el sub-diccionario del LLM
sys_prompt, usr_prompt = build_prompt(
    context_data_llm=payload_sistema["LLM_Tab_2_Contexto"], 
    user_role=rol_usuario, 
    scope=alcance_analisis
)

print("System Prompt configurado correctamente. Listo para enviar a Ollama.")

System Prompt configurado correctamente. Listo para enviar a Ollama.


## 8. Inferencia, latencia y tolerancia a fallos (RNF-02, RNF-07)
Se ejecuta la llamada al LLM local (Ollama) con temperatura baja (0.3) para respuestas deterministas.
Se mide la latencia total y se valida contra el umbral RNF-02.
En caso de fallo de conexión, se devuelve un mensaje de contingencia sin bloquear el dashboard.

In [4]:
# ==============================================================================
# 4. INFERENCIA, LATENCIA TOTAL Y TOLERANCIA A FALLOS (RNF-02, RNF-07)
# ==============================================================================
import time

def generate_recommendations(payload_completo: dict):
    # 1. Extracción de contexto
    scope = payload_completo['Configuracion']['Alcance']
    role = payload_completo['Configuracion']['Rol']
    
    # Llamamos a build_prompt pasándole solo el sub-diccionario del LLM
    system_p, user_p = build_prompt(payload_completo['LLM_Tab_2_Contexto'], role, scope)
    
    print(f"Iniciando solicitud al LLM local (Ollama) para {role} en {scope}...\n")
    print("⏳ [UI SIMULATION: Mostrando pantalla de 'Estado de Procesamiento Asíncrono' al usuario...]")
    
    start_time = time.time()
    
    try:
        # 2.Llamada en bloque (REST estándar) con Optimizaciones de Rendimiento
        response = ollama.chat(
            model=LLM_MODEL, 
            messages=[
                {'role': 'system', 'content': system_p},
                {'role': 'user', 'content': user_p}
            ],
            stream=False,
            options={
                "temperature": 0.3,       
                "num_ctx": 4096        
            }
        )
        
        end_time = time.time()
        total_latency = end_time - start_time
        texto_completo = response['message']['content']
        
        # 3. Validación Automática del RNF-02 (Tiempo Total < 10s)
        print("\n--- MÉTRICAS DE RENDIMIENTO ---")
        status_rnf02 = "✅ CUMPLE RNF-02" if total_latency <= 10.0 else "⚠️ EXCEDE RNF-02"
        print(f"⏱️ Tiempo total de procesamiento (Latencia Backend): {total_latency:.2f} segundos [{status_rnf02}]\n")
        
        print("✅ [UI SIMULATION: Transición al Dashboard Ejecutivo completada]")
        print("\n--- RECOMENDACIONES GENERADAS ---")
        print(texto_completo)
        
        return texto_completo
        
    except Exception as e:
        # 4. Tolerancia a fallos (RNF-07)
        end_time = time.time()
        print(f"\n❌ ERROR DE CONEXIÓN CON EL LLM (Latencia fallida: {end_time - start_time:.2f}s)")
        print(f"Detalle técnico: {str(e)}")
        
        mensaje_contingencia = (
            "⚠️ Sistema de recomendaciones temporalmente no disponible.\n"
            "El dashboard de métricas sigue operativo. Por favor, revise los "
            "semáforos de riesgo manualmente o contacte con soporte técnico."
        )
        print("\n--- RESPUESTA DE CONTINGENCIA ---")
        print(mensaje_contingencia)
        
        return mensaje_contingencia

# Ejecución
recomendaciones_finales = generate_recommendations(payload_sistema)

Iniciando solicitud al LLM local (Ollama) para Project Manager en Sprint...

⏳ [UI SIMULATION: Mostrando pantalla de 'Estado de Procesamiento Asíncrono' al usuario...]

--- MÉTRICAS DE RENDIMIENTO ---
⏱️ Tiempo total de procesamiento (Latencia Backend): 19.43 segundos [⚠️ EXCEDE RNF-02]

✅ [UI SIMULATION: Transición al Dashboard Ejecutivo completada]

--- RECOMENDACIONES GENERADAS ---
• Reasignar la tarea "PRO-301 'Configurar pipeline de CI/CD en el nuevo entorno de staging'" a un equipo más experimentado, ya que tiene un riesgo generalizado del 100% y un retraso previsto del 80%. Esto permitirá mitigar el impacto operativo inmediato y reducir la carga de trabajo.

• Priorizar la resolución de bloqueos en las tareas "PRO-301", "PRO-306" y "PRO-307", ya que tienen un conteo de bloqueadores (Blocker_Count) igual a 5. Esto garantizará el progreso en estas tareas críticas y reducirá la probabilidad de retraso.

• Repriorizar el backlog para enfocarse en las tareas con mayor riesgo y retras

## 9. Auditoría de calidad (LLM-as-a-judge)
Se utiliza un segundo modelo LLM como juez imparcial para verificar:
- **Coherencia** con los datos ML originales.
- **Accionabilidad** de las recomendaciones.
- **Transparencia** (citas de métricas).
- **Ausencia de contradicciones** internas.

Este mecanismo valida los criterios de calidad del módulo LLM definidos en el plan de gestión de la calidad.

In [5]:
# ==============================================================================
# 5. LLM-AS-A-JUDGE (Verificación del Plan de Calidad)
# ==============================================================================

def audit_quality(payload_completo: dict, generated_text: str) -> str:
    # Pasamos el mismo bloque de datos que vio el modelo
    original_context = json.dumps(payload_completo['LLM_Tab_2_Contexto'], ensure_ascii=False)
    
    audit_prompt = f"""
    Actúas como un Auditor de Calidad de Software. Evalúa el siguiente texto generado frente a los datos originales proporcionados.
    
    DATOS ORIGINALES: {original_context}
    TEXTO GENERADO: {generated_text}
    
    EVALÚA LOS SIGUIENTES CRITERIOS DE CALIDAD:
    1. Ausencia de contradicciones internas (Pasa/Falla).
    2. Coherencia con los datos (Pasa/Falla). ¿Se han respetado los nombres de las tareas y las variables como Blocker_Count?
    3. Recomendaciones accionables (Pasa/Falla). ¿Indica pasos operativos a seguir?
    4. Transparencia (Pasa/Falla). ¿Menciona los datos numéricos explícitamente para justificar la acción?
    
    Proporciona un reporte estricto y conciso de la evaluación en forma de lista.
    """
    
    try:
        evaluation = ollama.chat(model=LLM_MODEL, messages=[
            {'role': 'user', 'content': audit_prompt}
        ], options={'temperature': 0.0})
        return evaluation['message']['content']
    except Exception as e:
        return f"Error en la auditoría: {str(e)}"

print("\n--- REPORTE DE AUDITORÍA (LLM-as-a-judge) ---")
print(audit_quality(payload_sistema, recomendaciones_finales))


--- REPORTE DE AUDITORÍA (LLM-as-a-judge) ---
**Evaluación de Calidad**

1. **Ausencia de contradicciones internas**: Pasa
El texto generado no presenta contradicciones internas, ya que las recomendaciones se basan en los datos proporcionados y no hay conflictos entre ellas.

2. **Coherencia con los datos**: Pasa
La evaluación coincide con los datos originales. Los nombres de las tareas (PRO-301, PRO-306, etc.) son correctos, y la variable Blocker_Count se menciona correctamente como un indicador de riesgo.

3. **Recomendaciones accionables**: Pasa
El texto generado presenta recomendaciones claras y específicas para abordar los problemas identificados en las tareas PRO-301, PRO-306, y PRO-307. Estos pasos operativos a seguir son:
	* Reasignar la tarea "PRO-301" a un equipo más experimentado.
	* Priorizar la resolución de bloqueos en las tareas mencionadas.
	* Repriorizar el backlog para enfocarse en tareas con mayor riesgo y retraso.

4. **Transparencia**: Pasa
El texto generado propo